In [11]:
import pandas as pd
from dataload import *
from bpr import *

In [12]:
%load_ext autoreload
%autoreload 2
import importlib
%load_ext autoreload
%autoreload 2
import bpr
import dataload
importlib.reload(dataload)
importlib.reload(bpr)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<module 'bpr' from '/home/roee/Documents/git-repos/OPC/BPR/bpr.py'>

In [13]:
ml_ratings, ml_users, ml_movies = load_movielens_1m('/home/roee/Documents/git-repos/OPC/datasets/ml-1m')

In [14]:
myket_ratings, myket_users, myket_apps = load_myket('/home/roee/Documents/git-repos/OPC/datasets/myket')

In [15]:
ml = build_csr_from_interactions(
    interactions=ml_ratings[ml_ratings["rating"] >= 4][["user_id", "movie_id"]],
    user_col="user_id",
    item_col="movie_id",
    item_info=ml_movies,  # has title/genres
    assume_users_are_indices=False
)

In [16]:
mk = build_csr_from_interactions(
    interactions=myket_ratings[["user_id", "item_id"]],  # after your fix
    user_col="user_id",
    item_col="item_id",
    item_info=myket_apps,               # app_info_sample merged or just item_id list
    assume_users_are_indices=False
)

In [ ]:
# MovieLens
bpr_ml = BayesianPersonalizedRanking(
    factors=32, learning_rate=0.05, regularization=1e-4,
    epochs=30, mode="samples", samples_per_epoch=100_000, random_state=0
).fit(ml.X)

bpr_ml.save_embeddings(
    user_path='/home/roee/Documents/git-repos/OPC/BPR/embeddings/ml_user_factors.npy',
    item_path='/home/roee/Documents/git-repos/OPC/BPR/embeddings/ml_item_factors.npy'
)

In [17]:
save_user_interaction_counts('/home/roee/Documents/git-repos/OPC/BPR/embeddings/ml_user_interaction_counts.npy', ml_ratings, ml.user2idx, user_col='user_id')

array([ 53, 129,  51, ...,  20, 123, 341], shape=(6038,))

In [ ]:
# Myket
bpr_mk = BayesianPersonalizedRanking(
    factors=32, learning_rate=0.05, regularization=1e-4,
    epochs=30, mode="samples", samples_per_epoch=150_000, random_state=0
).fit(mk.X)

bpr_mk.save_embeddings(
    user_path='/home/roee/Documents/git-repos/OPC/BPR/embeddings/myket_user_factors.npy',
    item_path='/home/roee/Documents/git-repos/OPC/BPR/embeddings/myket_item_factors.npy'
)   

In [19]:
save_user_interaction_counts('/home/roee/Documents/git-repos/OPC/BPR/embeddings/myket_user_interaction_counts.npy', myket_ratings, mk.user2idx, user_col='user_id')

array([ 45,  33, 140, ..., 134,  80, 119], shape=(10000,))

In [ ]:
show_item_neighbors(
    model=bpr_ml,
    item_id=13,                      # movie_id
    item2idx=ml.item2idx,
    idx2item=ml.idx2item,
    item_info=ml.item_info,
    k=5,
    fields=["title", "genres"],
)

In [ ]:
show_item_neighbors(
    model=bpr_mk,
    item_id='ir.gamelogic.cutoff',                    # item_id
    item2idx=mk.item2idx,
    idx2item=mk.idx2item,
    item_info=mk.item_info,
    k=5,
    fields=["index", "category_en"]
)

In [21]:
ratings_msd, users_msd, items_msd = load_artistwise_dfs("../datasets/msd/msd_taste_profile.hdf5")
ratings_lastfm, users_lastfm, items_lastfm = load_artistwise_dfs("../datasets/lastfm/lastfm_360k.hdf5")

In [22]:
lastfm = build_csr_from_interactions(
    interactions=ratings_lastfm[["user_id", "item_id"]],
    user_col="user_id",
    item_col="item_id",
    item_info=items_lastfm,  # has title/genres
    assume_users_are_indices=False
)

msd = build_csr_from_interactions(
    interactions=ratings_msd[["user_id", "item_id"]],
    user_col="user_id",
    item_col="item_id",
    item_info=items_msd,  # has title/genres
    assume_users_are_indices=False
)

In [ ]:
bpr_lastfm = BayesianPersonalizedRanking(
    factors=32, learning_rate=0.1, regularization=1e-5,
    epochs=30, mode="samples", samples_per_epoch=250_000, random_state=0
).fit(lastfm.X)

bpr_lastfm.save_embeddings(
    user_path='/home/roee/Documents/git-repos/OPC/BPR/embeddings/lastfm_user_factors.npy',
    item_path='/home/roee/Documents/git-repos/OPC/BPR/embeddings/lastfm_item_factors.npy'
)

In [23]:
save_user_interaction_counts('/home/roee/Documents/git-repos/OPC/BPR/embeddings/lastfm_user_interaction_counts.npy', ratings_lastfm, lastfm.user2idx, user_col='user_id')

array([49, 51, 46, ..., 21, 50, 48], shape=(358868,))

In [ ]:
bpr_msd = BayesianPersonalizedRanking(
    factors=32, learning_rate=0.1, regularization=1e-5,
    epochs=50, mode="samples", samples_per_epoch=350_000, random_state=0
).fit(msd.X)

bpr_msd.save_embeddings(
    user_path='/home/roee/Documents/git-repos/OPC/BPR/embeddings/msd_user_factors.npy',
    item_path='/home/roee/Documents/git-repos/OPC/BPR/embeddings/msd_item_factors.npy'
)

In [24]:
save_user_interaction_counts('/home/roee/Documents/git-repos/OPC/BPR/embeddings/msd_user_interaction_counts.npy', ratings_msd, msd.user2idx, user_col='user_id')

array([11, 13, 11, ..., 15, 36, 28], shape=(1019318,))

In [ ]:
show_item_neighbors(
    model=bpr_lastfm,
    item_id='blue Öyster cult',                      # item_id
    item2idx=lastfm.item2idx,
    idx2item=lastfm.idx2item,
    item_info=lastfm.item_info,
    k=5,
    fields=[]
)

In [ ]:
show_item_neighbors(
    model=bpr_msd,
    item_id='Pearl Jam',                      # item_id
    item2idx=msd.item2idx,
    idx2item=msd.idx2item,
    item_info=msd.item_info,
    k=5,
    fields=[]
)

In [ ]:
ratings_anime, users_anime, items_anime = load_anime_dfs("../datasets/anime/")

anime = build_csr_from_interactions(
    interactions=ratings_anime[["user_id", "item_id"]],
    user_col="user_id",
    item_col="item_id",
    item_info=items_anime,  # has title/genres
    assume_users_are_indices=False
)

bpr_anime = BayesianPersonalizedRanking(
    factors=32, learning_rate=0.1, regularization=1e-5,
    epochs=15, mode="samples", samples_per_epoch=250_000, random_state=0
).fit(anime.X)

bpr_anime.save_embeddings(  
    user_path='/home/roee/Documents/git-repos/OPC/BPR/embeddings/anime_user_factors.npy',
    item_path='/home/roee/Documents/git-repos/OPC/BPR/embeddings/anime_item_factors.npy'
)

In [27]:
save_user_interaction_counts('/home/roee/Documents/git-repos/OPC/BPR/embeddings/anime_user_interaction_counts.npy', ratings_anime, anime.user2idx, user_col='user_id')

array([153,   3,  74, ...,   1, 188,   2], shape=(73417,))

In [ ]:
show_item_neighbors(
    model=bpr_anime,
    item_id=100,                      # item_id
    item2idx=anime.item2idx,
    idx2item=anime.idx2item,
    item_info=anime.item_info,
    k=5,
    fields=["title", "genres"]    
)